# CyTOF Data Loading and Preparation

Loads raw CyTOF data exported from OMIQ and prepares the annotated cell matrix used for correction.

**Workflow:**
1. Load the OMIQ metadata file (sample sheet)
2. Format it into the standard metadata table expected by the pipeline
3. Configure the pipeline (`CyTOFContext`)
4. Call `load_data_from_omiq` to get the annotated cell matrix
5. Inspect and save the matrix for use in `Correction.ipynb`

The generated CSV contains all cells, annotated  
with `sample_name`, `plate`, `dataset`, `panel`, `condition`, and all marker columns.

In [ ]:
import sys
import os
import json
import glob
import re
import pandas as pd

# Add the project root to the Python path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src import load_data_from_omiq, get_config, CyTOFContext, read_cytof_file
from src.utils import harmonize_marker


## Step 1 — Load the OMIQ metadata file

OMIQ exports a sample sheet (Excel or CSV) that lists the filenames, sample labels,  
and any experimental annotations. Load it here and display its structure.

In [ ]:
# Dataset settings: only this block should usually change between databases.
study_name = "example_study"
dataset_to_load = "D0"
panel_to_load = None

base_dir = "../data"
data_dir = os.path.join(base_dir, "omiq_exported_data")
output_dir = "../results"

omiq_metadata_path = os.path.join(base_dir, "metadata.xlsx")
metadata_sheet_name = 0

metadata_col_map = {
    "file_name": "Filename",
    "plate": None,          # None = infer from filename, e.g. Plate1_sample.csv
    "sample_name": None,    # None = infer from filename after removing PlateX_
    "dataset": None,        # None = use dataset_to_load
    "panel": None,          # None = use "Panel1"
    "condition": "Ex-vivo treatment",
}

ref_plate = "Plate1"
anchors = {dataset_to_load: "ANCHOR_SAMPLE"}

# Optional overrides. Leave marker list as None to infer it from the first data file.
celltype_col = None
true_markers_override = None
celltype_gating_override = None

df_omiq = pd.read_excel(omiq_metadata_path, sheet_name=metadata_sheet_name)

print(f"Shape: {df_omiq.shape}")
print(f"Columns: {list(df_omiq.columns)}")
display(df_omiq.head())


## Step 2 — Format into the standard metadata table

The pipeline expects a DataFrame with these columns:

| Column | Description |
|---|---|
| `file_name` | Filename as stored in `data_dir` (e.g. `Plate1_Sample01.csv`) |
| `sample_name` | Biological sample identifier |
| `plate` | Plate/batch label (must match `ref_plate` in config) |
| `dataset` | Dataset/timepoint identifier (e.g. `D0`) |
| `panel` | Panel name (e.g. `Panel1`) |
| `condition` | Experimental condition (optional but recommended) |

Below is an example of how to derive `plate` and `sample_name` from the OMIQ filename  
when the naming convention is `<Plate>_<SampleName>.csv`.

In [ ]:
my_metadata = pd.DataFrame()

# file_name: convert .fcs -> .csv to match exported files
def get_meta_col(target, default=None):
    col = metadata_col_map.get(target)
    return df_omiq[col] if col and col in df_omiq.columns else default

file_col = metadata_col_map.get("file_name")
file_series = get_meta_col("file_name")
if file_series is None:
    raise KeyError("metadata_col_map['file_name'] must point to the metadata filename column")
my_metadata['file_name'] = file_series.astype(str).str.replace('.fcs', '.csv', regex=False)

# plate: first PlateX token in filename unless metadata_col_map provides a column
def parse_plate(filename):
    match = re.search(r"(Plate\d+)", str(filename), flags=re.IGNORECASE)
    return match.group(1) if match else ref_plate

plate_series = get_meta_col("plate")
my_metadata['plate'] = plate_series.astype(str) if plate_series is not None else my_metadata['file_name'].apply(parse_plate)

# sample_name: strip plate prefix and file extension unless metadata_col_map provides a column
def parse_sample_name(row):
    name = os.path.splitext(str(row['file_name']))[0]
    prefix = str(row['plate']) + '_'
    return name[len(prefix):] if name.startswith(prefix) else name

sample_series = get_meta_col("sample_name")
my_metadata['sample_name'] = sample_series.astype(str) if sample_series is not None else my_metadata.apply(parse_sample_name, axis=1)

# Fixed or metadata-provided columns for this cohort
dataset_series = get_meta_col("dataset")
panel_series = get_meta_col("panel")
condition_series = get_meta_col("condition")

def assign_condition(row) -> str:
    """Map database-specific labels to broad biological/treatment states."""
    filename = str(row.get(file_col, "")).lower()
    subject = str(row.get("Subject", "")).lower()
    treatment = str(row.get("Ex-vivo treatment", "")).strip().lower()
    notes = str(row.get("Notes", "")).lower()

    if "healthy" in filename or subject.startswith("hbm") or subject == "anchor" or "anchor" in notes:
        return "healthy"
    if treatment in {"none", "untreated", "dx untreated", "unt"} or "untreated" in filename:
        return "untreated"
    if treatment in {"", "nan", "unknown"}:
        return "unknown"
    return "treated"


my_metadata['dataset'] = dataset_series.astype(str) if dataset_series is not None else dataset_to_load
my_metadata['panel'] = panel_series.astype(str) if panel_series is not None else "Panel1"

if {"Subject", "Ex-vivo treatment"}.issubset(df_omiq.columns):
    my_metadata['condition'] = df_omiq.apply(assign_condition, axis=1)
else:
    my_metadata['condition'] = condition_series.fillna("unknown").astype(str) if condition_series is not None else "unknown"

print("Formatted metadata:")
display(my_metadata.head(10))
print(f"\nPlates found: {sorted(my_metadata['plate'].unique())}")
print(f"Samples found: {sorted(my_metadata['sample_name'].unique())[:20]} ...")

# Quick file-match check before loading all cells.
data_files = {os.path.basename(p) for p in glob.glob(os.path.join(data_dir, '*')) if p.lower().endswith(('.csv', '.fcs'))}
missing_on_disk = sorted(set(my_metadata['file_name']) - data_files)
missing_in_metadata = sorted(data_files - set(my_metadata['file_name']))
print(f"\nFiles in metadata : {my_metadata['file_name'].nunique()}")
print(f"Files on disk     : {len(data_files)}")
print(f"Missing on disk   : {len(missing_on_disk)}")
print(f"Missing in metadata: {len(missing_in_metadata)}")


In [ ]:
my_metadata["condition"].unique()


## Step 3 — Configure the pipeline

The `CyTOFContext` centralises all pipeline settings.  
Set the marker panel, metadata columns, anchors, and directories for the study.

In [ ]:
# Infer markers and cell type column from the first data file, reusing package readers/helpers.
input_files = sorted(
    p for p in glob.glob(os.path.join(data_dir, '*'))
    if p.lower().endswith(('.csv', '.fcs'))
)
if not input_files:
    raise FileNotFoundError(f"No CSV/FCS files found in {data_dir}")

first_df_header = read_cytof_file(input_files[0], nrows=0) if input_files[0].lower().endswith('.csv') else read_cytof_file(input_files[0])
raw_columns = list(first_df_header.columns)

if celltype_col is None:
    for candidate in ['cell_type', 'celltypes', 'OmiqFilter', 'Population', 'Gate']:
        if candidate in raw_columns:
            celltype_col = candidate
            break

ignored_cols = {'Time', 'Event_length', celltype_col, None}
inferred_markers = [
    harmonize_marker(c)
    for c in raw_columns
    if c not in ignored_cols and not str(c).startswith('Unnamed:')
]
inferred_markers = list(dict.fromkeys(inferred_markers))
true_markers = true_markers_override or inferred_markers

celltype_gating = celltype_gating_override or [
    "T-cells",
    "B-cells",
    "megakaryocytic",
    "megakaryocytic-like",
    "myeloid-enriched",
    "bulk_leukocytes",
    "immature-like",
    "CD45low/198Pt-high",
    "CD45low",
    "CD45inter",
    "CD45pos",
    "CD45high",
    "cells",
    "alive",
    "CD45neg"

]

my_config = {
    # Directory containing the OMIQ-exported CSV/FCS files
    "data_dir": data_dir,

    # Where corrected results will be written (only used by run_cytof_correction)
    "output_dir": output_dir,

    # The plate whose anchor cells define the reference distribution
    "ref_plate": ref_plate,

    # Anchor sample name per dataset - must appear in every plate
    "anchors": anchors,

    # Columns that carry metadata (not biological signal)
    "meta_cols": [
    "Event_length",
    "Time",
    "dataset",
    "panel",
    "plate",
    "sample_name",
    "cell_type",
    "condition"
    ],

    # OmiqFilter gate names used for cell type assignment
    "celltypes": celltype_gating,

    # Cell types to apply time-drift correction to
    "celltype2correct": celltype_gating,

    # The marker used as a time-drift reference (e.g. DNA intercalator)
    "stable_marker": "DNA",

    # Biological markers to include in batch correction
    "true_markers": true_markers,

    # Metal -> marker name mapping (for OMIQ channel renaming, leave empty if not needed)
    "metal_to_marker": {},

    # Gating column names in the OMIQ export that define CD45+ populations
    "celltype_gating": celltype_gating
}

config = get_config(config_dict=my_config)
ctx    = CyTOFContext(config)
print("CyTOFContext created successfully")
print(f"  data_dir    : {ctx.data_dir}")
print(f"  ref_plate   : {ctx.ref_plate}")
print(f"  anchors     : {ctx.anchors}")
print(f"  celltype_col: {celltype_col}")
print(f"  markers ({len(true_markers)}): {true_markers}")


## Step 4 — Load and annotate the cell matrix

`load_data_from_omiq` reads every matching file, harmonizes marker names,  
applies the marker schema, assigns cell types, and annotates each cell with  
the metadata from `my_metadata`.

Use the `dataset` and `panel` arguments to restrict loading to a subset.

In [ ]:
# celltype_col: OMIQ CSV column containing cell-type labels.
# Set celltype_col=None for files that use OmiqFilter binary flag columns.
matrix, markers = load_data_from_omiq(
    data_dir     = ctx.data_dir,
    metadata_df  = my_metadata,
    ctx          = ctx,
    dataset      = dataset_to_load,   # set to None to load all datasets
    panel        = panel_to_load,     # set to 'Panel1' etc. to restrict to one panel
    celltype_col = celltype_col,
)

print(f"\nMatrix shape : {matrix.shape}")
print(f"Markers ({len(markers)}): {markers}")


In [ ]:
# Normalize path-like OMIQ labels such as "89Y_CD45+/T-cells" -> "T-cells".
if "celltypes" in matrix.columns:
    matrix["celltypes"] = matrix["celltypes"].astype(str).str.split("/").str[-1]
    print(matrix.celltypes.unique())

if "celltypes" in matrix.columns and ctx.celltype_gating:
    matrix = matrix[matrix["celltypes"].isin(ctx.celltype_gating)]
    print(f"\nMatrix shape after cell type filtering: {matrix.shape}")
else:
    print("\nNo cell-type labels/gating available; keeping all cells and skipping time correction.")


In [ ]:
matrix.columns


## Step 5 — Explore the annotated matrix

The matrix contains one row per cell. Metadata columns come first, followed by  
biological marker columns. Check the structure before saving.

In [ ]:
# Preview
display(matrix.head())

# Summary statistics per plate
print("\nCells per plate:")
display(matrix.groupby('plate')['sample_name'].value_counts().rename('n_cells').reset_index())

# Cell type distribution
if 'celltypes' in matrix.columns:
    print("\nCell type distribution:")
    display(matrix['celltypes'].value_counts())

## Step 6 — Save the matrix

Save the annotated matrix to disk so that `Correction.ipynb` can load it directly  
without re-processing the raw files.

In [ ]:
results_dir = ctx.output_dir
os.makedirs(results_dir, exist_ok=True)

study_dataset = matrix["dataset"].iloc[0] if "dataset" in matrix.columns else dataset_to_load
study_panel   = matrix["panel"].iloc[0]   if "panel"   in matrix.columns else "unknown"
file_tag = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(study_dataset))

# Save the annotated matrix
matrix_path = os.path.join(results_dir, f"matrix_{file_tag}.csv")
matrix.to_csv(matrix_path, index=False)
print(f"Matrix saved  : {matrix_path}")

# Save the marker list
markers_path = os.path.join(results_dir, f"markers_{file_tag}.json")
with open(markers_path, "w") as f:
    json.dump(markers, f)
print(f"Markers saved : {markers_path}")

# Study config - single source of truth for all other notebooks
matrix_corrected_path    = os.path.join(results_dir, f"matrix_{file_tag}_corrected.csv")
matrix_intermediate_path = os.path.join(results_dir, f"matrix_{file_tag}_intermediate.csv")

study_config = {
    "study_name":   study_name,
    "dataset":      study_dataset,
    "panel":        study_panel,
    "cytof_config": my_config,
    "outputs": {
        "matrix_path":              matrix_path,
        "markers_path":             markers_path,
        "matrix_corrected_path":    matrix_corrected_path,
        "matrix_intermediate_path": matrix_intermediate_path,
    },
}

study_config_path = os.path.join(results_dir, f"study_config_{file_tag}.json")
with open(study_config_path, "w") as f:
    json.dump(study_config, f, indent=2)
print(f"Study config  : {study_config_path}")
print(f"\nIn Correction.ipynb set:")
print(f'  study_config_path = "{study_config_path}"')
